# 🔗 Notebook 3: Relation Extraction

Extract semantic relations (triples) from text using:
- **Hearst Patterns** for is-a (hypernym) relations
- **Dependency Parsing** for subject-verb-object triples

---

In [ ]:
!pip install -q spacy scikit-learn PyMuPDF
!python -m spacy download en_core_web_sm -q
!git clone https://github.com/YOUR_USERNAME/ontology-builder.git 2>/dev/null || true
import sys; sys.path.insert(0, 'ontology-builder')

## 1. Load Data from Previous Notebooks

In [ ]:
import json, os
from src.preprocessor import TextPreprocessor, TextSegment
from src.concept_extractor import ConceptExtractor, Concept

# Demo text if no previous output
demo_texts = [
    """Drugs such as Aspirin, Ibuprofen, and Acetaminophen are analgesics.
    Aspirin treats headaches and reduces inflammation.
    Ibuprofen is a type of nonsteroidal anti-inflammatory drug.
    Headaches are a common pain disorder.
    Migraine is a severe form of headache.
    Acetaminophen reduces fever and relieves pain.
    Nonsteroidal anti-inflammatory drugs, including Naproxen and Ibuprofen, 
    reduce prostaglandin synthesis.""",
    
    """Pain management involves pharmacological and non-pharmacological approaches.
    Pharmacological treatments include opioids, NSAIDs, and local anesthetics.
    Chronic pain is a long-lasting pain condition.
    Acute pain is a short-term pain response to injury.
    The WHO pain ladder recommends a step-wise approach to pain management."""
]

pp = TextPreprocessor(min_segment_length=15)
segments = []
for t in demo_texts:
    segments.extend(pp.preprocess(t))

ce = ConceptExtractor(min_frequency=1)
concepts = ce.extract(segments)
print(f'Loaded {len(segments)} segments, {len(concepts)} concepts')

## 2. Hearst Pattern Extraction

Hearst patterns capture taxonomic (is-a) relations from natural language.

In [ ]:
import re

# Demonstrate patterns manually
test_sentences = [
    'Drugs such as Aspirin, Ibuprofen, and Acetaminophen are analgesics.',
    'Ibuprofen is a type of nonsteroidal anti-inflammatory drug.',
    'Nonsteroidal anti-inflammatory drugs, including Naproxen and Ibuprofen, are common.',
    'Migraine is a severe form of headache.',
]

pattern_such_as = r'(\b[A-Z][a-z]+(?:\s+[a-z]+)*)\s+such\s+as\s+((?:[A-Z][a-z]+(?:\s+[a-z]+)*(?:,\s*)?)+(?:\s+and\s+[A-Z][a-z]+(?:\s+[a-z]+)*)?)'

print('Hearst Pattern Matches:')
print('=' * 60)
for sent in test_sentences:
    matches = re.findall(pattern_such_as, sent)
    for m in matches:
        hypernym = m[0]
        hyponyms = re.split(r',\s*|\s+and\s+', m[1])
        for h in hyponyms:
            if h.strip():
                print(f'  {h.strip()} ──isA──▶ {hypernym}')

## 3. Dependency-Based Triple Extraction

In [ ]:
import spacy

nlp = spacy.load('en_core_web_sm')

def extract_svo_triples(text):
    """Extract Subject-Verb-Object triples from dependency parse."""
    doc = nlp(text)
    triples = []
    
    for sent in doc.sents:
        for token in sent:
            if token.pos_ != 'VERB':
                continue
            
            subjects = [c for c in token.children if c.dep_ in ('nsubj', 'nsubjpass')]
            objects = [c for c in token.children if c.dep_ in ('dobj', 'attr')]
            
            for subj in subjects:
                for obj in objects:
                    triples.append({
                        'subject': subj.text,
                        'verb': token.lemma_,
                        'object': obj.text,
                        'sentence': sent.text.strip()
                    })
    return triples

# Test
test_text = 'Aspirin treats headaches. Ibuprofen reduces inflammation. Migraine causes severe pain.'
triples = extract_svo_triples(test_text)

print('Extracted SVO Triples:')
print('=' * 60)
for t in triples:
    print(f"  {t['subject']} ──{t['verb']}──▶ {t['object']}")
    print(f"    Source: {t['sentence']}")

## 4. Full Relation Extraction Pipeline

In [ ]:
from src.relation_extractor import RelationExtractor

rel_extractor = RelationExtractor(
    use_hearst=True,
    use_dependency=True,
    min_confidence=0.3
)

relations = rel_extractor.extract(segments, concepts)

print(f'\nExtracted {len(relations)} relations:\n')
print(f'{"Subject":<25} {"Predicate":<20} {"Object":<25} {"Conf":<6} {"Type"}')
print('=' * 90)
for r in relations[:25]:
    print(f'{r.subject:<25} {r.predicate:<20} {r.object:<25} {r.confidence:<6.2f} {r.relation_type}')

## 5. Visualize Relations as a Graph

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

G = nx.DiGraph()
for r in relations:
    G.add_edge(r.subject, r.object, label=r.predicate)

if G.nodes:
    plt.figure(figsize=(14, 10))
    pos = nx.spring_layout(G, k=2, seed=42)
    
    nx.draw_networkx_nodes(G, pos, node_size=600, node_color='#4A90D9', alpha=0.8)
    nx.draw_networkx_edges(G, pos, arrows=True, arrowsize=15, alpha=0.6)
    nx.draw_networkx_labels(G, pos, font_size=8, font_weight='bold')
    
    edge_labels = {(u, v): d['label'] for u, v, d in G.edges(data=True)}
    nx.draw_networkx_edge_labels(G, pos, edge_labels, font_size=6)
    
    plt.title('Extracted Relations Graph', fontsize=14)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('No relations to visualize.')

## 6. Save Relations

In [ ]:
os.makedirs('output', exist_ok=True)

relations_data = [
    {'subject': r.subject, 'predicate': r.predicate, 'object': r.object,
     'confidence': r.confidence, 'type': r.relation_type}
    for r in relations
]

with open('output/relations.json', 'w') as f:
    json.dump(relations_data, f, indent=2)

print(f'Saved {len(relations_data)} relations to output/relations.json')

---
**Next:** [Notebook 4 — Ontology Building](04_ontology_building.ipynb)